# 10-Worked Examples from Lectures

In [1]:
import sympy as sp
import sympy.physics.mechanics as spm
import sympy.physics.vector as spv
from sympy.physics.vector.printing import init_vprinting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import IPython

# Import IPython display for proper LaTeX formatting
from IPython.display import display, Math, Markdown

# Initialize symbols
sp.init_printing()

# Enable dot notation printing for dynamicsymbols
init_vprinting(use_latex="mathjax")

HALF = sp.S.Half

In [2]:
def reference_frame(frame: str, x=r"\imath", y=r"\jmath", z=r"k") -> spm.ReferenceFrame:
    """Create a SymPy reference frame with custom basis vector labels.

    Parameters
    ----------
    frame : str
        The name of the reference frame.
    x, y, z : str
        Labels for the basis vectors.
    """
    return spm.ReferenceFrame(
        frame,
        latexs=(
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{x}}}",
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{y}}}",
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{z}}}",
        ),
    )


def reference_frame_circular(name: str, angle=r"theta") -> spm.ReferenceFrame:
    """Create a circular reference frame with radial and angular basis labels.

    Parameters
    ----------
    name : str
        Name of the new reference frame.
    angle : str, optional
        Symbol or label used for the angular basis vector, by default "theta".
    """
    return reference_frame(name, x=r"r", y=rf"\{angle}", z=r"e_z")

## 31.3 WE-Find the Moment of Inertia of a Disc from a Falling Mass

| System Diagram                                                       | Inertial Reference Frame                                    |
|----------------------------------------------------------------------|-------------------------------------------------------------|
| ![Moment of Inertia of a Disc](../figures/WE3103-01-MassFalling.jpg) | ![Inertial Frame](../figures/WE3103-03-MassFalling-x-y.jpg) |


| Pulley Force Diagram                                               | Rotating Frame                                                |
|--------------------------------------------------------------------|---------------------------------------------------------------|
| ![Pull Force Diagram](../figures/WE3103-05-MassFalling-pulley.jpg) | ![Rotating Frame](../figures/WE3103-04-MassFalling-A-x-y.jpg) |

In [3]:
(R, # Radius of the disc
 m, # Mass of block
 h, # Height of block,
 Icm, # Moment of inertia of disc about the center of mass
 T, # Tension in the string
 g # Acceleration due to gravity
) = sp.symbols('R m h I_cm T g', real=True, positive=True)

N = reference_frame('N')
A = reference_frame("A")
theta = spv.dynamicsymbols("theta", real=True, positive=True)
A.orient_axis(N, N.z, theta)

R_vec = R * A.x
R_vel = R_vec.dt(N)
R_acc = R_vel.dt(N)

T_vec = T * (-N.x)
R_cm_T_vec = R * N.y
tau_pulley_cm = R_cm_T_vec.cross(T_vec)

display(
    Math(
        r"\text{Torque on the pulley about its center of mass: } \tau_{cm} ="
        f"{spv.vlatex(tau_pulley_cm)}"
    )
)

vec_alpha = A.ang_acc_in(N)
display(
    Math(
        r"\text{Angular acceleration vector: } \vec \alpha =" 
        f"{spv.vlatex(vec_alpha)}"
    )
)


N2LReqn = sp.Eq(tau_pulley_cm.dot(N.z), Icm * vec_alpha.dot(N.z))
display(
    Math(
        r"\text{N2LR torque and angular acceleration: }"
        f"{spv.vlatex(N2LReqn)}"
    )
)

a = sp.Symbol("a", real=True)
alpha = a / R
T_solution = sp.solve(N2LReqn, Icm, dict=True)[0]
Icm_solution = T_solution[Icm].subs({theta.diff().diff(): a/R})
display(
    Math(
        r"\text{Moment of inertia of the pulley about its center of mass: } I_{cm} ="
        rf"\boxed{{{spv.vlatex(Icm_solution)}}}"
    )
)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

![Mass descending](../figures/WE3103-06-MassFalling-Mass.jpg)

In [4]:
Forces_on_m = T * N.x - m * g * N.x
N2Leqn = sp.Eq(Forces_on_m.dot(N.x), -m * R * theta.diff().diff())
display(Math(r"\text{Newton's 2nd Law for the mass: }" f"{spv.vlatex(N2Leqn)}"))

solution = sp.solve([N2Leqn, N2LReqn], [T, Icm], dict=True)[0]

display(Math(r"\text{N2L Solution: }" f"{spv.vlatex(solution)}"))

solution_simplify = {
    k: sp.simplify(v.subs(theta.diff().diff(), a / R)) for k, v in solution.items()
}
display(Math(r"\text{Simplified N2L Solution: }" f"{spv.vlatex(solution_simplify)}"))

Icm_solution = solution_simplify[Icm]
display(
    Math(
        r"\text{Simplified Moment of inertia of the pulley about its center of mass: } I_{cm} ="
        rf"\boxed{{{spv.vlatex(Icm_solution)}}}"
    )
)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>